# 02 — 單一 Engine：轉換 · 檢視 · Batch 推論輸出

以 FP16 engine 為例，完整走一遍從 ONNX 轉換到逐張圖分類輸出的流程。

| 步驟 | 內容 |
|------|------|
| 1 | 路徑設定 |
| 2 | ONNX 模型 shape 確認 |
| 3 | trtexec 轉換 → FP16 engine |
| 4 | TensorRT Python API 引擎檢視（選用）|
| 5 | Batch 讀圖 + 推論 + 結果輸出 |

## 1. 路徑設定

In [7]:
import sys, importlib
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import src.env
importlib.reload(src.env)
from src.env import *

ENGINES_DIR.mkdir(exist_ok=True)

WARMUP_MS  = 200   # quick warmup for conversion check
MAX_IMAGES = 50    # max images for batch inference

print("Settings ready.")
print(f"TEST_DATASET : {TEST_DATASET}")

Settings ready.
TEST_DATASET : D:\data\Test_dataset


## 2. ONNX 模型 Shape 確認

In [8]:
import onnx

model = onnx.load(str(ONNX_MODEL))
onnx.checker.check_model(model)

def fmt_shape(tensor_type):
    dims = tensor_type.shape.dim
    return [d.dim_value if d.dim_value > 0 else (d.dim_param or "?") for d in dims]

print(f"Opset : {model.opset_import[0].version}")

print("\nInputs:")
for t in model.graph.input:
    print(f"  {t.name:40s}  {fmt_shape(t.type.tensor_type)}")

print("\nOutputs:")
for t in model.graph.output:
    print(f"  {t.name:40s}  {fmt_shape(t.type.tensor_type)}")

_in    = model.graph.input[0]
_shape = fmt_shape(_in.type.tensor_type)
MODEL_H = _shape[2] if isinstance(_shape[2], int) else 448
MODEL_W = _shape[3] if isinstance(_shape[3], int) else 448
MODEL_C = _shape[1] if isinstance(_shape[1], int) else 3
print(f"\nInference shape: [1, {MODEL_C}, {MODEL_H}, {MODEL_W}]")

Opset : 12

Inputs:
  images                                    [1, 3, 448, 448]

Outputs:
  output                                    [1, 6]

Inference shape: [1, 3, 448, 448]


## 3. trtexec 轉換 → FP16 Engine
如果 `engines/H_fp16.engine` 已存在，可跳過本格直接執行後續步驟。

In [9]:
import subprocess, time

if ENGINE_FP16.exists():
    print(f"Engine already exists: {ENGINE_FP16}  (delete it to rebuild)")
else:
    cmd = [
        str(TRTEXEC),
        f"--onnx={ONNX_MODEL}",
        f"--saveEngine={ENGINE_FP16}",
        f"--warmUp={WARMUP_MS}",
        "--fp16",
        "--skipInference",   # build only; no latency benchmark
    ]
    print("Building FP16 engine (first run takes a few minutes) ...")
    print("CMD:", " ".join(cmd), "\n")

    t0   = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0

    if proc.returncode != 0:
        print("=== STDERR ===")
        print(proc.stderr[-3000:])
    else:
        for line in proc.stdout.splitlines():
            if any(kw in line for kw in ["[I]", "[W]", "[E]", "Saving"]):
                print(line)

    print(f"\nElapsed: {elapsed:.1f}s")
    print(f"Engine saved: {ENGINE_FP16}  exists={ENGINE_FP16.exists()}")

Engine already exists: engines\H_fp16.engine  (delete it to rebuild)


## 4. TensorRT Python API — 引擎結構檢視（選用）

需先安裝 TensorRT Python bindings：
```
uv pip install "C:/Users/edisonhsieh/Downloads/TensorRT-10.8.0.43.Windows.win10.cuda-12.8/TensorRT-10.8.0.43/python/tensorrt-10.8.0.43-cp312-none-win_amd64.whl"
```
未安裝時本格自動跳過。

In [10]:
try:
    import tensorrt as trt
    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
    print(f"TensorRT Python : {trt.__version__}")

    with open(ENGINE_FP16, "rb") as f:
        engine = trt.Runtime(TRT_LOGGER).deserialize_cuda_engine(f.read())

    n = engine.num_io_tensors
    print(f"\nEngine: {ENGINE_FP16}  ({n} tensors)")
    for i in range(n):
        name  = engine.get_tensor_name(i)
        shape = list(engine.get_tensor_shape(name))
        dtype = engine.get_tensor_dtype(name)
        mode  = engine.get_tensor_mode(name).name
        print(f"  [{mode:6s}] {name}: shape={shape}, dtype={dtype}")

except ImportError:
    print("tensorrt not installed — skipping (see cell header for install command).")

TensorRT Python : 10.8.0.43

Engine: engines\H_fp16.engine  (2 tensors)
  [INPUT ] images: shape=[1, 3, 448, 448], dtype=DataType.FLOAT
  [OUTPUT] output: shape=[1, 6], dtype=DataType.FLOAT


## 5. Batch 讀圖 + 推論 + 結果輸出

從 `TEST_DATASET` 讀取最多 `MAX_IMAGES` 張圖，使用 **ONNXRuntime GPU**（或 TRT FP16 如果已安裝 tensorrt Python package），  
輸出每張圖的路徑、預測類別（label）和信心分數。

In [11]:
import numpy as np
import cv2
import onnxruntime as ort
from tqdm import tqdm
import pandas as pd

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
CLASS_NAMES = [str(i) for i in range(1, 10)]   # classes "1"–"9"; adjust if needed

image_paths = [
    p for p in TEST_DATASET.rglob("*")
    if p.suffix.lower() in IMG_EXTS and p.is_file()
][:MAX_IMAGES]
print(f"Found {len(image_paths)} images in {TEST_DATASET}")

if not image_paths:
    print("No images found — skipping inference. Check TEST_DATASET path in .env")
    df = pd.DataFrame(columns=["file", "path", "label", "confidence"])
else:
    def softmax(x: np.ndarray) -> np.ndarray:
        x = x.astype(np.float32) - x.max()
        e = np.exp(x)
        return e / e.sum()

    def preprocess(path, h, w):
        img = cv2.imread(str(path))
        if img is None:
            return None
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (w, h))
        img = img.astype(np.float32) / 255.0
        return np.transpose(img, (2, 0, 1))[None, ...]   # [1, C, H, W]

    # Force CUDA only — specifying both providers causes silent CPU fallback
    session = ort.InferenceSession(
        str(ONNX_MODEL),
        providers=["CUDAExecutionProvider"]
    )
    inp_name = session.get_inputs()[0].name
    out_name = session.get_outputs()[0].name
    provider = session.get_providers()[0].replace("ExecutionProvider", "")
    print(f"Active provider : {provider}")

    records = []
    for p in tqdm(image_paths, desc="Inference"):
        x = preprocess(p, MODEL_H, MODEL_W)
        if x is None:
            records.append({"file": p.name, "path": str(p), "label": "read_error", "confidence": float("nan")})
            continue

        out = session.run([out_name], {inp_name: x})[0].reshape(-1)
        if out.min() < 0 or not np.isclose(out.sum(), 1.0, atol=1e-3):
            out = softmax(out)

        idx   = int(np.argmax(out))
        label = CLASS_NAMES[idx] if idx < len(CLASS_NAMES) else str(idx)
        records.append({
            "file":       p.name,
            "path":       str(p.relative_to(TEST_DATASET)),
            "label":      label,
            "confidence": float(out[idx]),
        })

    df = pd.DataFrame(records)
    print(f"\nBatch inference complete — {len(df)} images  (provider: {provider})")
    print(df[["file", "label", "confidence"]].to_string(index=False, float_format="{:.4f}".format))

Found 50 images in D:\data\Test_dataset
Active provider : CPU


Inference: 100%|██████████| 50/50 [00:03<00:00, 12.71it/s]


Batch inference complete — 50 images  (provider: CPU)
                                              file label  confidence
                     Defect0000001 (1503)_34tr.jpg     3      1.0000
                     Defect0000001 (1505)_34tr.jpg     3      1.0000
                     Defect0000001 (1511)_34tr.jpg     3      1.0000
                        Defect0000002 (2)_12tr.jpg     3      1.0000
                        Defect0000002 (6)_34tr.jpg     3      1.0000
                        Defect0000002 (8)_12tr.jpg     3      1.0000
             0202_DPNG_Defect0000001 (18)_12tr.jpg     3      1.0000
            0202_DPNG_Defect0000001 (25)_34rec.jpg     3      1.0000
             0202_DPNG_Defect0000001 (28)_34tr.jpg     3      1.0000
                  0202_DPNG_Defect0000001_12tr.jpg     3      1.0000
              0202_DPNG_Defect0000002 (3)_12tr.jpg     3      1.0000
              0202_DPNG_Defect0000002 (7)_34tr.jpg     3      1.0000
                                     (10)_12tr.j

## 6. 結果統計

In [12]:
if df.empty:
    print("No results to summarise.")
else:
    print("Label distribution:")
    print(df["label"].value_counts().to_string())

    print(f"\nMean confidence : {df['confidence'].mean():.4f}")
    print(f"Min  confidence : {df['confidence'].min():.4f}")
    print(f"Max  confidence : {df['confidence'].max():.4f}")

    print("\nLowest confidence predictions:")
    print(df.nsmallest(5, "confidence")[["file", "label", "confidence"]].to_string(index=False, float_format="{:.4f}".format))

Label distribution:
label
3    50

Mean confidence : 1.0000
Min  confidence : 1.0000
Max  confidence : 1.0000

Lowest confidence predictions:
                         file label  confidence
Defect0000001 (1503)_34tr.jpg     3      1.0000
Defect0000001 (1505)_34tr.jpg     3      1.0000
Defect0000001 (1511)_34tr.jpg     3      1.0000
   Defect0000002 (2)_12tr.jpg     3      1.0000
   Defect0000002 (6)_34tr.jpg     3      1.0000
